# Bronze Layer Validation

Purpose:
- Validate successful Kafka ingestion
- Inspect schema
- Perform basic data quality checks
- Understand raw data before Silver transformations

Input:
Bronze Delta Table

Output:
Validated Bronze Dataset

In [0]:
%run ./00_Project_Setup

# Log Guardian - Project Setup

This notebook contains all project-level configurations required by the Log Guardian streaming pipeline.

Responsibilities:
- Import required libraries
- Configure Kafka connection
- Configure Aiven authentication
- Load SSL certificate
- Define common project paths
- Define reusable variables

No data processing is performed in this notebook.

openstack-normal1,openstack-normal2,openstack-abnormal


In [0]:
bronze_df = spark.read.format("delta").load(
    "/Volumes/log-analytics/bronze/key_volume/bronze_delta_v2"
)

In [0]:
# Basic Info
print("="*60)
print("BRONZE DATASET")
print("="*60)

print(f"Rows    : {bronze_df.count()}")
print(f"Columns : {len(bronze_df.columns)}")

bronze_df.printSchema()

BRONZE DATASET
Rows    : 281900
Columns : 22
root
 |-- event_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- dataset_source: string (nullable = true)
 |-- log_file: string (nullable = true)
 |-- service: string (nullable = true)
 |-- process_id: string (nullable = true)
 |-- log_level: string (nullable = true)
 |-- request_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- project_id: string (nullable = true)
 |-- instance_id: string (nullable = true)
 |-- client_ip: string (nullable = true)
 |-- http_method: string (nullable = true)
 |-- http_path: string (nullable = true)
 |-- message: string (nullable = true)
 |-- status_code: integer (nullable = true)
 |-- response_time: double (nullable = true)
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
# Preview 
bronze_df.show(10, truncate=False)

+------------------------------------+-----------------------+--------------+------------------+---------------------------+----------+---------+------------------------------------+-------+----------+-----------+---------+-----------+---------+-----------------------------------------------------------------------------------------------------------------+-----------+-------------+------------------+---------------+------------+-----------------------+----------------------+
|event_id                            |timestamp              |dataset_source|log_file          |service                    |process_id|log_level|request_id                          |user_id|project_id|instance_id|client_ip|http_method|http_path|message                                                                                                          |status_code|response_time|kafka_topic       |kafka_partition|kafka_offset|kafka_timestamp        |ingestion_timestamp   |
+------------------------------------+

In [0]:
# Null Value Checking
from pyspark.sql.functions import *

null_counts = bronze_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in bronze_df.columns
])

null_counts.show(truncate = False)

+--------+---------+--------------+--------+-------+----------+---------+----------+-------+----------+-----------+---------+-----------+---------+-------+-----------+-------------+-----------+---------------+------------+---------------+-------------------+
|event_id|timestamp|dataset_source|log_file|service|process_id|log_level|request_id|user_id|project_id|instance_id|client_ip|http_method|http_path|message|status_code|response_time|kafka_topic|kafka_partition|kafka_offset|kafka_timestamp|ingestion_timestamp|
+--------+---------+--------------+--------+-------+----------+---------+----------+-------+----------+-----------+---------+-----------+---------+-------+-----------+-------------+-----------+---------------+------------+---------------+-------------------+
|0       |0        |0             |0       |0      |0         |0        |22576     |114067 |114067    |206976     |138666   |138666     |138666   |0      |138666     |138666       |0          |0              |0           |0

In [0]:
# Duplicate Events
duplicates = (
    bronze_df.groupBy("event_id")
        .count()
        .filter("count > 1")
)

print("Duplicate Event IDs:", duplicates.count())
duplicates.show()

Duplicate Event IDs: 3
+--------------------+-----+
|            event_id|count|
+--------------------+-----+
|d2d59581-6a72-43e...|    2|
|c9b2d89e-959d-4ab...|    2|
|8b4ef017-dff1-4df...|    2|
+--------------------+-----+



In [0]:
# Service Distribution
bronze_df.groupBy("service")\
        .count()\
        .orderBy("service")\
        .show()

+--------------------+------+
|             service| count|
+--------------------+------+
|keystonemiddlewar...|     3|
|nova.api.openstac...|  3001|
|nova.api.openstac...|  2999|
| nova.compute.claims| 24008|
|nova.compute.manager| 36017|
|nova.compute.reso...|  8192|
|nova.metadata.wsg...| 29423|
|nova.osapi_comput...|113811|
|nova.scheduler.ho...|  1020|
|nova.virt.libvirt...| 15020|
|nova.virt.libvirt...| 48142|
|oslo_service.peri...|   264|
+--------------------+------+



In [0]:
# HTTP Status Codes
bronze_df.groupBy("status_code")\
        .count()\
        .orderBy("status_code")\
        .show()

+-----------+------+
|status_code| count|
+-----------+------+
|       NULL|138666|
|        200|131483|
|        202|  3002|
|        204|  2999|
|        404|  5749|
|        503|     1|
+-----------+------+



In [0]:
# kafka validation
bronze_df.groupBy(
    "kafka_topic",
    "kafka_partition"
).count().show()

+------------------+---------------+------+
|       kafka_topic|kafka_partition| count|
+------------------+---------------+------+
|openstack-abnormal|              0| 48074|
| openstack-normal1|              1|101760|
| openstack-normal2|              1|132066|
+------------------+---------------+------+



In [0]:
# offset validation
bronze_df.select(
    min("kafka_offset").alias("Min Offset"),
    max("kafka_offset").alias("Max Offset")
).show()

+----------+----------+
|Min Offset|Max Offset|
+----------+----------+
|         0|    132065|
+----------+----------+



In [0]:
# Response Time Statistics
bronze_df.describe("response_time").show()

+-------+-------------------+
|summary|      response_time|
+-------+-------------------+
|  count|             143234|
|   mean|0.23405903514179702|
| stddev|0.10258797117478562|
|    min|           5.369E-4|
|    max|          1.2951999|
+-------+-------------------+



In [0]:
# Timestamp Range
bronze_df.select(
    min("kafka_timestamp"),
    max("kafka_timestamp")
).show()

+--------------------+--------------------+
|min(kafka_timestamp)|max(kafka_timestamp)|
+--------------------+--------------------+
|2026-07-19 13:33:...|2026-07-22 12:43:...|
+--------------------+--------------------+



In [0]:
# Temporary View
bronze_df.createOrReplaceTempView("bronze_logs")

In [0]:
%sql
-- SQL Validation
SELECT
    log_level,
    COUNT(*) AS total_logs
FROM bronze_logs
GROUP BY log_level
ORDER BY total_logs DESC;

log_level,total_logs
INFO,277119
WARNING,4515
ERROR,265
CRITICAL,1


In [0]:
# Summary
print("="*60)
print("Bronze Layer Validation Completed")
print("="*60)

print(f"Rows Loaded : {bronze_df.count()}")
print(f"Columns     : {len(bronze_df.columns)}")
print("Data Quality Checks : Completed")
print("Bronze Layer Status : SUCCESS")

Bronze Layer Validation Completed
Rows Loaded : 281900
Columns     : 22
Data Quality Checks : Completed
Bronze Layer Status : SUCCESS
